# 模糊综合评价

**原理**：将模糊数学与综合评价结合，通过隶属函数量化定性边界，适合处理"好/中/差"这类模糊评价。

**步骤**：确定因素集和评语集 → 建立隶属度矩阵 → 合成算子 → 综合评价向量 → 得分


In [ ]:
import numpy as np
import sys; sys.path.insert(0, '..')
from src.algorithms.fuzzy_comprehensive_evaluation import (
    validate_weights, validate_membership_matrix,
    apply_operator, normalize_b,
    fuzzy_comprehensive_evaluate, build_membership_matrix,
    FuzzyOperator, MembershipMethod
)
from src.utils.plot import Plotter


## 示例：教学质量评价

**因素集** = {教学内容, 教学方法, 教学态度, 教学效果}  
**评语集** = {优秀, 良好, 一般, 较差}  

隶属度矩阵 R 表示每个因素对各评语的隶属程度：


In [ ]:
# 隶属度矩阵：4 因素 × 4 评语
R = np.array([
    [0.3, 0.4, 0.2, 0.1],   # 教学内容
    [0.2, 0.3, 0.4, 0.1],   # 教学方法
    [0.4, 0.3, 0.2, 0.1],   # 教学态度
    [0.1, 0.3, 0.4, 0.2],   # 教学效果
])

validate_membership_matrix(R, n_indicators=4, n_grades=4)
print("隶属度矩阵 R：")
print(R)

# 解读：教学内容对"优秀"的隶属度为 0.3，对"良好"为 0.4...


In [ ]:
# 因素权重和各等级分数
factors = ["教学内容", "教学方法", "教学态度", "教学效果"]
grades = ["优秀", "良好", "一般", "较差"]
scores = [95, 82, 67, 50]  # 各等级对应的百分制分数

weights = validate_weights(np.array([0.25, 0.25, 0.30, 0.20]))
print(f"因素权重：{np.round(weights, 4)}")


In [ ]:
# 对比四种合成算子
print("=== 不同合成算子的评价结果 ===\n")
for op in FuzzyOperator:
    B = apply_operator(weights, R, op)
    B = normalize_b(B)
    result = fuzzy_comprehensive_evaluate(weights, R, scores, grades, op)
    print(f"{op.value}：")
    print(f"  B（综合隶属度）= {np.round(B, 4)}")
    print(f"  综合得分 = {result['score']:.2f}，等级 = {result['grade']}")
    print()


In [ ]:
# 可视化
fig, axes = Plotter.subplots(1, 2, figsize=(13, 5))

# 隶属度矩阵热力图
Plotter.heatmap(R, row_labels=factors, col_labels=grades,
                ax=axes[0], title="隶属度矩阵 R", fmt=".2f")

# 各因素对不同等级的隶属度曲线
for factor in factors:
    i = factors.index(factor)
    Plotter.line(R[i], label=factor, ax=axes[1], marker="o")
axes[1].set_title("各因素隶属度分布")
axes[1].set_xticks(range(len(grades)))
axes[1].set_xticklabels(grades)
axes[1].legend()

Plotter.save("../data/fuzzy_result.png")
print("图片已保存至 data/fuzzy_result.png")


In [ ]:
# 进阶：用梯形隶属函数自动生成隶属度矩阵
matrix = np.array([
    [85.0, 90.0, 78.0, 92.0],
    [76.0, 88.0, 85.0, 85.0],
    [90.0, 82.0, 92.0, 78.0],
])

R_auto = build_membership_matrix(
    matrix,
    grade_names=["优良", "中等", "较差"],
    kinds=[1, 1, 1, 1],
    method=MembershipMethod.TRAPEZOID,
)
print("自动生成的隶属度矩阵形状:", R_auto.shape)
print("\n方案 1 对各等级的隶属度：")
print(np.round(R_auto[0], 3))
